In [2]:
# Mount GG drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

In [4]:
!pip install sentence-transformers qdrant-client langchain pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 24.5 MB/s eta 0:00:00


In [5]:
df = pd.read_csv("/content/drive/MyDrive/news_info/financial_news.csv")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   source   10000 non-null  object
 1   title    10000 non-null  object
 2   url      10000 non-null  object
 3   content  10000 non-null  object
dtypes: object(4)
memory usage: 312.6+ KB


In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pandas as pd

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = []

for _, row in df.iterrows():
    text_to_split = row["title"] + ". " + row["content"]

    for chunk in text_splitter.split_text(text_to_split):
        chunks.append({
            "source": row["source"],
            "title": row["title"],
            "url": row["url"],
            "text": chunk
        })

df_chunks = pd.DataFrame(chunks)

print(df_chunks.head())
print("Số lượng chunk:", len(df_chunks))

      source                             title  \
0  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
1  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
2  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
3  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
4  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   

                                                 url  \
0  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
1  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
2  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
3  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
4  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   

                                                text  
0  Chứng khoán vượt ngưỡng 400 điểm. Ủy ban Chứng...  
1  với thông tin phải công bố theo quy định pháp ...  
2  bán niên năm 2021, Báo cáo tình hình sử dụng n...  
3  trai là Nguyễn Đức Thụy (1976) - "bầu Thụy" và...  
4  email:[email protected], hotline: 086 508 6899...  
Số lượng chunk: 60246


In [14]:
from google.colab import userdata
import os

# Lấy token Hugging Face từ Colab Secrets
os.environ["HF_TOKEN_RAG"] = userdata.get("HF_TOKEN_RAG")

In [17]:
from sentence_transformers import SentenceTransformer
import pandas as pd

df_chunks['text'] = df_chunks['text'].fillna("")

df_chunks = df_chunks[df_chunks['text'].str.strip().str.len() >= 50]
model = SentenceTransformer("BAAI/bge-m3", use_auth_token=os.environ["HF_TOKEN_RAG"])
texts = df_chunks['text'].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64
)
df_chunks['embedding'] = embeddings.tolist()
print(df_chunks.head())
print("Số lượng chunk:", len(df_chunks))
print("Vector embedding chiều:", len(df_chunks["embedding"][0]))

/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


Batches:   0%|          | 0/936 [00:00<?, ?it/s]

      source                             title  \
0  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
1  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
2  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
3  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   
4  thanhnien  Chứng khoán vượt ngưỡng 400 điểm   

                                                 url  \
0  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
1  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
2  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
3  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   
4  https://thanhnien.vn/chung-khoan-vuot-nguong-4...   

                                                text  \
0  Chứng khoán vượt ngưỡng 400 điểm. Ủy ban Chứng...   
1  với thông tin phải công bố theo quy định pháp ...   
2  bán niên năm 2021, Báo cáo tình hình sử dụng n...   
3  trai là Nguyễn Đức Thụy (1976) - "bầu Thụy" và...   
4  email:[email protected], hotline: 086 508 6899...   

                         

In [18]:
df_chunks.to_parquet("/content/financial_news_embedded.parquet")